In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import os
os.makedirs("pictures", exist_ok=True)

In [2]:
df = pd.read_csv(
    "19.inter_amplicon_distances.tsv",
    sep="\t"
)

valid_chroms = [f"chr{i}" for i in range(1, 23)] + ["chrX", "chrY"]

df_clean = df[df["Chromosome"].isin(valid_chroms)]

df_clean

#N_intervals-Количество расстояний между соседними ампликонами на этой хромосоме.
#Mean_distance (bp)- Среднее расстояние (в парах оснований) между соседними ампликонами на данной хромосоме.
#Min (bp)- Минимальное расстояние между двумя соседними ампликонами.
#Max (bp)- Максимальное расстояние между соседними ампликонами.

,Chromosome,N_intervals,Mean_distance(bp),Min(bp),Max(bp)
0,chr1,2715,91479.3,1,18201201
1,chr10,1649,80866.2,2,2650274
2,chr11,1942,69387.5,1,1418364
3,chr12,1793,74166.3,1,2863221
4,chr13,1362,71852.4,3,1435542
5,chr14,1113,81467.1,2,2319588
11,chr15,914,92772.5,4,2918398
13,chr16,747,119087.2,1,10403681
15,chr17,626,132452.4,2,4010283
17,chr18,972,82391.8,3,1751378


In [3]:
#решаем проблему с разделителем
df_dist =  pd.read_csv(
    "19.inter_amplicon_just_distances.tsv",
    sep=r"\s+",
    engine="python",
    names=["Chromosome", "Distance_bp"],
    header=0
)

valid_chroms = [f"chr{i}" for i in range(1, 23)] + ["chrX", "chrY"]

df_dist_24 = df_dist[df_dist["Chromosome"].isin(valid_chroms)]

summary = (
    df_dist_24
    .groupby("Chromosome")["Distance_bp"]
    .agg(
        N_intervals="count",
        Mean_bp="mean",
        Median_bp="median",
        Min_bp="min",
        Max_bp="max",
        Std_bp="std"
    )
    .reset_index()
)

df_dist_24 

,Chromosome,Distance_bp
0,chr1,52
1,chr1,7468
2,chr1,361199
3,chr1,29
4,chr1,1184
...,...,...
39324,chrY,23
39325,chrY,1633633
39326,chrY,3
39327,chrY,1087874


In [4]:
plt.figure(figsize=(10,4))
sns.boxplot(
    data=df_dist_24,
    x="Chromosome",
    y="Distance_bp",
    color="#8172B3"
)
plt.yscale("log")
plt.xlabel("Chromosome")
plt.ylabel("Inter-amplicon distance (bp, log scale)")
plt.title("Distribution of inter-amplicon distances per chromosome")
plt.tight_layout()
plt.savefig("pictures/Distribution of inter-amplicon distances per chromosome.png", dpi=300)
plt.close()

In [5]:
import math

chroms = sorted(df_dist_24["Chromosome"].unique())

n = len(chroms)
cols = 6
rows = math.ceil(n / cols)

fig, axes = plt.subplots(rows, cols, figsize=(15, 2.8 * rows))
axes = axes.flatten()

for ax, chr_ in zip(axes, chroms):
    data = df_dist_24.loc[
        (df_dist_24["Chromosome"] == chr_) &
        (df_dist_24["Distance_bp"] > 0),
        "Distance_bp"
    ]

    # логарифмические бины
    bins = np.logspace(
        np.log10(data.min()),
        np.log10(data.max()),
        25
    )
    
    ax.hist(
    data,
    bins=bins,
    color="#8172B3",
    edgecolor="black",
    linewidth=0.6,
    alpha=0.85
    )

    ax.set_xscale("log")
    ax.set_title(chr_, fontsize=9)
    ax.tick_params(labelsize=7)

    ax.set_xlabel("bp", fontsize=7)
    ax.set_ylabel("Count", fontsize=7)

# убрать пустые панели
for ax in axes[n:]:
    ax.axis("off")

fig.suptitle(
    "Distribution of inter-amplicon distances per chromosome",
    y=1.02
)

plt.tight_layout()
plt.savefig("pictures/hist distances per chromosome.png", dpi=300)
plt.close()

In [6]:
#распределение глубин покрытия для покрытых участков
df_depth = pd.read_csv(
    "19.covered_positions_depth.tsv",
    sep="\t",
    names=["Chromosome", "Position", "Depth"]
)

df_depth.head()

,Chromosome,Position,Depth
0,chr1,64470,12
1,chr1,64471,12
2,chr1,64472,12
3,chr1,64473,12
4,chr1,64474,12


In [7]:
#Общие сведение о покрытии
depth_summary = df_depth["Depth"].agg(
    N_positions="count",
    Mean_depth="mean",
    Median_depth="median",
    Min_depth="min",
    Max_depth="max"
)

depth_summary

N_positions     4.936736e+06
Mean_depth      5.820658e+01
Median_depth    1.800000e+01
Min_depth       1.000000e+00
Max_depth       1.424800e+04
Name: Depth, dtype: float64

In [8]:
#Таблица с распределением покрытия по бинам
bins = [1, 10, 50, 100, 500, 1000, df_depth["Depth"].max()]
labels = [
    "1–10",
    "10–50",
    "50–100",
    "100–500",
    "500–1000",
    ">1000"
]

df_depth["Depth_bin"] = pd.cut(
    df_depth["Depth"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

bin_table = (
    df_depth["Depth_bin"]
    .value_counts()
    .sort_index()
    .reset_index()
    .rename(columns={
        "index": "Depth_range",
        "Depth_bin": "N_positions"
    })
)

bin_table

,N_positions,count
0,1–10,1977869
1,10–50,1592900
2,50–100,509829
3,100–500,851321
4,500–1000,3231
5,>1000,1586


In [9]:
valid_chroms = [f"chr{i}" for i in range(1, 23)] + ["chrX", "chrY"]

df_depth_clean = df_depth[
    df_depth["Chromosome"].isin(valid_chroms)
]

chrom_depth = (
    df_depth_clean
    .groupby("Chromosome")["Depth"]
    .agg(
        N_positions="count",
        Mean_depth="mean",
        Median_depth="median"
    )
    .reset_index()
)

chrom_depth

,Chromosome,N_positions,Mean_depth,Median_depth
0,chr1,336812,56.299974,18.0
1,chr10,206473,59.078214,17.0
2,chr11,244032,52.092758,18.0
3,chr12,224259,63.611373,20.0
4,chr13,170281,66.748216,19.0
5,chr14,139830,58.825002,20.0
6,chr15,112554,56.602502,18.0
7,chr16,93398,58.856828,20.0
8,chr17,78940,66.155713,20.0
9,chr18,121534,57.041100,18.0


In [10]:
bins = np.logspace(
    np.log10(df_depth_clean["Depth"].min()),
    np.log10(df_depth_clean["Depth"].max()),
    50
)

plt.figure(figsize=(6,4))
plt.hist(
    df_depth_clean["Depth"],
    bins=bins,
    color="#8172B3",
    edgecolor="black",
    linewidth=0.6,
    alpha=0.85
)

plt.xscale("log")
plt.xlabel("Depth of coverage (log scale)")
plt.ylabel("Number of positions")
plt.title("Distribution of coverage depth for covered positions")
plt.tight_layout()
plt.savefig("pictures/coverage_depth_distribution.png", dpi=300)
plt.close()